In [38]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# ----------------------------------------
# Configuración
# ----------------------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN"
output_dir = os.path.join(input_dir, "Suavizado_spline")
indice = "NDVI"
col_fecha = "FECHA"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# Procesar cada archivo
# ----------------------------------------
for file in os.listdir(input_dir):
    if file.endswith("_sin_periodo_nieve_filtr.csv"):
        nombre = file.replace("_sin_periodo_nieve_filtr.csv", "")
        ruta_csv = os.path.join(input_dir, file)

        try:
            df = pd.read_csv(ruta_csv, sep=";", encoding="latin1")

            if col_fecha not in df.columns or indice not in df.columns:
                print(f"⚠️  Saltado {nombre}: columnas faltantes")
                continue

            df[col_fecha] = df[col_fecha].astype(str).str.strip()
            df[indice] = df[indice].astype(str).str.strip()

            df = df[df[col_fecha] != '']
            df = df[df[indice] != '']

            df['FECHA_dt'] = pd.to_datetime(df[col_fecha], format='%Y-%m-%d', errors='coerce')
            df[indice] = pd.to_numeric(df[indice], errors='coerce')
            df = df.dropna(subset=['FECHA_dt', indice])

            if df.empty:
                print(f"⚠️  Saltado {nombre}: sin datos válidos")
                continue

            df = df.sort_values('FECHA_dt')
            x = (df['FECHA_dt'] - df['FECHA_dt'].min()).dt.days.values
            y = df[indice].values

            mask = ~np.isnan(x) & ~np.isnan(y)
            x_clean, y_clean = x[mask], y[mask]

            if len(x_clean) < 5:
                print(f"⚠️  Saltado {nombre}: muy pocos puntos para suavizar")
                continue

            spline = UnivariateSpline(x_clean, y_clean, s=0.0025 * len(x_clean))
            x_pred = np.linspace(x_clean.min(), x_clean.max(), 500)
            y_pred = spline(x_pred)
            fechas_pred = df['FECHA_dt'].min() + pd.to_timedelta(x_pred, unit='D')

            # ✅ Corrección: eliminar `.dt`
            df_suavizado = pd.DataFrame({
                'FECHA': fechas_pred.strftime('%Y-%m-%d'),
                f"{indice}_suavizado": y_pred
            })

            # Gráfico
            plt.figure(figsize=(10, 5))
            plt.scatter(df['FECHA_dt'], df[indice], color='gray', alpha=0.4, label='Original', s=10)
            plt.plot(fechas_pred, y_pred, color='blue', label='Spline suavizado')
            plt.title(f"{indice} – Spline – {nombre.replace('_', ' ')}")
            plt.xlabel("Fecha")
            plt.ylabel(indice)
            plt.grid(True)
            plt.legend()
            plt.tight_layout()

            ruta_png = os.path.join(output_dir, f"{indice}_{nombre}_Spline.png")
            plt.savefig(ruta_png, dpi=150)
            plt.close()

            ruta_csv_suavizado = os.path.join(output_dir, f"{indice}_{nombre}_Spline.csv")
            df_suavizado.to_csv(ruta_csv_suavizado, index=False, sep=";")

            print(f"✅ Procesado {nombre}: gráfico y CSV guardados")

        except Exception as e:
            print(f"❌ Error procesando {nombre}: {e}")

print("\n🎯 Proceso finalizado: Suavizado Spline aplicado a todas las turberas.")


✅ Procesado Bassa_Nera: gráfico y CSV guardados
✅ Procesado Clots_de_Rialba: gráfico y CSV guardados
⚠️  Saltado Estanyeres: sin datos válidos
✅ Procesado Plan_deth_Horn: gráfico y CSV guardados
✅ Procesado Pla_de_Sartari: gráfico y CSV guardados
✅ Procesado Pleta_de_Molieres: gráfico y CSV guardados

🎯 Proceso finalizado: Suavizado Spline aplicado a todas las turberas.
